# Preprocessing
This notebook is the first step in the project workflow. 

Run it from top to bottom to load and validate the raw data, handle missing and exact duplicate records, decode category codes using `Dataset/raw/codetable.txt`, and save `Dataset/processed/preprocessed.csv` for later notebooks.

### 1. Load and inspect the raw data
Read the whitespace-delimited source file. Common missing-value markers are treated as nulls. The path fallback supports running from either the project root or the `notebooks` folder.

In [24]:
from pathlib import Path

import re
import numpy as np
import pandas as pd

# Set it so pandas display all columns when printing the dataframe, only to make it easier to look at.
pd.set_option('display.max_columns', None)
raw_folder = Path('Dataset/raw')


# Find the raw data folder
if not raw_folder.exists():
    raw_folder = Path('../Dataset/raw')
if not raw_folder.exists():
    raise FileNotFoundError('Could not find Dataset/raw')


# Initializes the paths for raw data, and the code table used to preprocess it.
raw_data_path = raw_folder / 'SouthGermanCredit.asc'
codetable_path = raw_folder / 'codetable.txt'


# Checks if the data and codetable is present
if not raw_data_path.exists():
    raise FileNotFoundError(f'Raw data not found in the folder: {raw_data_path}')
if not codetable_path.exists():
    raise FileNotFoundError(f'Codetable not found in the folder: {codetable_path}')


# Initialize the possible markers for missing values in the raw dataset
missing_markers = ['?', 'NA', 'N/A', 'NaN', 'nan', 'NULL', 'null', 'None', 'none', 'missing', '-', '--']



In [25]:
# Read the dataset
original_df = pd.read_csv(
    raw_data_path,
    sep=r'\s+',
    na_values=missing_markers
)

# Makes a copy of the original data, so we can preserve the original version and use it when needed.
df = original_df.copy()

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

Shape: (1000, 21)
Columns: ['laufkont', 'laufzeit', 'moral', 'verw', 'hoehe', 'sparkont', 'beszeit', 'rate', 'famges', 'buerge', 'wohnzeit', 'verm', 'alter', 'weitkred', 'wohn', 'bishkred', 'beruf', 'pers', 'telef', 'gastarb', 'kredit']


In [26]:
display(df.head())
df.info()

,laufkont,laufzeit,moral,verw,hoehe,sparkont,beszeit,rate,famges,buerge,wohnzeit,verm,alter,weitkred,wohn,bishkred,beruf,pers,telef,gastarb,kredit
0,1,18,4,2,1049,1,2,4,2,1,4,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,2,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,4,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,2,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,4,2,38,1,2,2,2,2,1,1,1


<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   laufkont  1000 non-null   int64
 1   laufzeit  1000 non-null   int64
 2   moral     1000 non-null   int64
 3   verw      1000 non-null   int64
 4   hoehe     1000 non-null   int64
 5   sparkont  1000 non-null   int64
 6   beszeit   1000 non-null   int64
 7   rate      1000 non-null   int64
 8   famges    1000 non-null   int64
 9   buerge    1000 non-null   int64
 10  wohnzeit  1000 non-null   int64
 11  verm      1000 non-null   int64
 12  alter     1000 non-null   int64
 13  weitkred  1000 non-null   int64
 14  wohn      1000 non-null   int64
 15  bishkred  1000 non-null   int64
 16  beruf     1000 non-null   int64
 17  pers      1000 non-null   int64
 18  telef     1000 non-null   int64
 19  gastarb   1000 non-null   int64
 20  kredit    1000 non-null   int64
dtypes: int64(21)
memory usage: 164.2 KB


## 2. Rename the columns from German to English

In [27]:
# We rename it based on the information from codetable.txt
rename_map = {
    'laufkont': 'status',
    'laufzeit': 'duration',
    'moral': 'credit_history',
    'verw': 'purpose',
    'hoehe': 'amount',
    'sparkont': 'savings',
    'beszeit': 'employment_duration',
    'rate': 'installment_rate',
    'famges': 'personal_status_sex',
    'buerge': 'other_debtors',
    'wohnzeit': 'present_residence',
    'verm': 'property',
    'alter': 'age',
    'weitkred': 'other_installment_plans',
    'wohn': 'housing',
    'bishkred': 'number_credits',
    'beruf': 'job',
    'pers': 'people_liable',
    'telef': 'telephone',
    'gastarb': 'foreign_worker',
    'kredit': 'credit_risk',
}

# Actual renaming part 
df = df.rename(columns=rename_map)
display(df.head())

,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,1,18,4,2,1049,1,2,4,2,1,4,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,2,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,4,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,2,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,4,2,38,1,2,2,2,2,1,1,1


## 3. Read the code table and identify feature types

The code table is at Dataset/processed/codetable.txt. We will us it to map the categorical features and predicted variable to their corresponding meaning.

In [28]:
# Initializes the regex patterns to read the codetable.txt
# For example, 
# -> " $`laufkont = status`       "  will match the pattern for section_pattern
# -> " 1 : no checking account    "  will match the pattern for code_pattern
section_pattern = re.compile(r"^\$`[^=]+=\s*([^`]+)`$")
code_pattern = re.compile(r"^\s*(-?\d+)\s*:\s*(.*?)\s*$")


# Dictionary for hold the category mappings for each columns. Current Column is to keep track which column is being mapped right now.
category_mappings = {}
current_column = None

# Loops through the entire codetable.txt file and reads through each line.
for line in codetable_path.read_text(encoding='utf-8').splitlines():
    
    # Checks if the current line is a section header
    section_match = section_pattern.match(line)
    if section_match:
        current_column = section_match.group(1).strip()
        category_mappings[current_column] = {}
        continue
    
    # Checks if the current line is a code mapping line
    code_match = code_pattern.match(line)
    if current_column is not None and code_match:
        code = int(code_match.group(1))
        label = code_match.group(2).strip()
        category_mappings[current_column][code] = label

# Maps the category mappings to only contain columns that have categories in the codetable.txt
# Numericals Columns dont contain any code matches in the loop before, so there is no mapping for them.
category_mappings = {
    column: mapping
    for column, mapping in category_mappings.items()
    if mapping
}
    
    
    

In [29]:
# Just making sure that the features are correctly categorized as numerical, nominal, and ordinal
numerical_features = ['duration', 'amount', 'age']

nominal_features = [
    'status', 'credit_history', 'purpose', 'savings', 'personal_status_sex',
    'other_debtors', 'other_installment_plans', 'housing', 'people_liable',
    'telephone', 'foreign_worker',
]

ordinal_features = [
    'employment_duration', 'installment_rate', 'present_residence',
    'property', 'number_credits', 'job',
]

categorical_features = nominal_features + ordinal_features

target = 'credit_risk'

# This just checked if all the categorical features and the target are mapped correctly, according to the codetabke.txt
# Just a sanity check
required_mappings = set(categorical_features + [target])
missing_mappings = sorted(required_mappings - category_mappings.keys())
if missing_mappings:
    raise ValueError(f'No category mappings found in codetable.txt for: {missing_mappings}')



# Creates a dataframe to display the feature data types
feature_types = pd.DataFrame({
    'feature': numerical_features + nominal_features + ordinal_features,
    'type': ['Numerical'] * len(numerical_features) + ['Nominal categorical'] * len(nominal_features) + ['Ordinal categorical'] * len(ordinal_features),
})

display(df.dtypes.to_frame('raw_dtype'))
display(feature_types)


,raw_dtype
status,int64
duration,int64
credit_history,int64
purpose,int64
amount,int64
savings,int64
employment_duration,int64
installment_rate,int64
personal_status_sex,int64
other_debtors,int64


,feature,type
0,duration,Numerical
1,amount,Numerical
2,age,Numerical
3,status,Nominal categorical
4,credit_history,Nominal categorical
5,purpose,Nominal categorical
6,savings,Nominal categorical
7,personal_status_sex,Nominal categorical
8,other_debtors,Nominal categorical
9,other_installment_plans,Nominal categorical


In [30]:
# Prints the category mappings for each categorical feature and the target.
for column in categorical_features + [target]:
    print(f'\n{column}')
    display(pd.Series(category_mappings[column], name='label').rename_axis('code').to_frame())


status


,label
code,
1,no checking account
2,... < 0 DM
3,0<= ... < 200 DM
4,... >= 200 DM / salary for at least 1 year



credit_history


,label
code,
0,delay in paying off in the past
1,critical account/other credits elsewhere
2,no credits taken/all credits paid back duly
3,existing credits paid back duly till now
4,all credits at this bank paid back duly



purpose


,label
code,
0,others
1,car (new)
2,car (used)
3,furniture/equipment
4,radio/television
5,domestic appliances
6,repairs
7,education
8,vacation


... (26 outputs left out)


job


,label
code,
1,unemployed/unskilled - non-resident
2,unskilled - resident
3,skilled employee/official
4,manager/self-empl./highly qualif. employee



credit_risk


,label
code,
0,bad
1,good


## 4. Check and handle missing values

The data source states that the dataset doesnt have any missing values, so we check if that is the case.

After Checking, the daatasest does indeed not have any missing values, so any handling is not required for null values.

In [31]:
missing_before = df.isna().sum()
display(missing_before.to_frame('missing_values'))


,missing_values
status,0
duration,0
credit_history,0
purpose,0
amount,0
savings,0
employment_duration,0
installment_rate,0
personal_status_sex,0
other_debtors,0


## 5. Check and remove exact duplicate records

In [32]:
duplicate_source = original_df.loc[df.index]
duplicate_mask = duplicate_source.duplicated(keep='first')
duplicate_count = int(duplicate_mask.sum())

print('Duplicate rows:', duplicate_count)



Duplicate rows: 0


## 6. Validation of Categorical and Numerical Data & Target Variable

In [33]:
# Categorical Data Validation
category_checks = []

for column in categorical_features + [target]:
    mapping = category_mappings[column]
    observed_codes = set(df[column].dropna().unique())
    allowed_codes = set(mapping)
    category_checks.append({
        'feature': column,
        'invalid_codes': sorted(observed_codes - allowed_codes),
        'unused_documented_codes': sorted(allowed_codes - observed_codes),
    })

category_validation = pd.DataFrame(category_checks)
display(category_validation)



# Numerical Data Validation
numerical_validation = pd.DataFrame({
    'minimum': df[numerical_features].min(),
    'maximum': df[numerical_features].max(),
    'non_finite_values': (~np.isfinite(df[numerical_features])).sum(),
    'non_positive_values': (df[numerical_features] <= 0).sum(),
})
display(numerical_validation)


# Ratio for the Predicted Variable
print('Target counts (0 = bad, 1 = good):')
display(df[target].value_counts().sort_index().to_frame('count'))

,feature,invalid_codes,unused_documented_codes
0,status,[],[]
1,credit_history,[],[]
2,purpose,[],[7]
3,savings,[],[]
4,personal_status_sex,[],[]
5,other_debtors,[],[]
6,other_installment_plans,[],[]
7,housing,[],[]
8,people_liable,[],[]
9,telephone,[],[]


,minimum,maximum,non_finite_values,non_positive_values
duration,4,72,0,0
amount,250,18424,0,0
age,19,75,0,0


Target counts (0 = bad, 1 = good):


,count
credit_risk,
0,300
1,700


## 7. Decode categories and save the preprocessed dataset

We previously made the categoty mappings with 'category_mappings'. Now we just need to replace valid categorical codes, including the target codes, with their established mappings.

In [34]:
# Makes a copy of the dataframe to hold the preprocessed data, then map the categorical features and target to their mappings
preprocessed_df = df.copy()

for column in categorical_features + [target]:
    mapping = category_mappings[column]
    decoded_values = preprocessed_df[column].map(mapping)
    
    # Only checking if there are any unmapped category values
    if decoded_values.isna().any():
        raise ValueError(f'Unmapped category values remain in {column}; refusing to save.')
    
    preprocessed_df[column] = pd.Categorical(
        decoded_values,
        categories=list(mapping.values()),
        ordered=column in ordinal_features,
    )

# Saving the preprocessed Dataset
processed_folder = raw_folder.parent / 'processed'
processed_folder.mkdir(parents=True, exist_ok=True)
 
preprocessed_path = processed_folder / 'preprocessed.csv'
preprocessed_df.to_csv(preprocessed_path, index=False)

print('Preprocessed dataset shape :', preprocessed_df.shape)
print('Saved to                   :', preprocessed_path.resolve())
display(preprocessed_df.head())

Preprocessed dataset shape : (1000, 21)
Saved to                   : /home/jason/Documents/Projects/CollegeProjects/CreditRiskClassificationProject/Dataset/processed/preprocessed.csv


,status,duration,credit_history,purpose,amount,savings,employment_duration,installment_rate,personal_status_sex,other_debtors,present_residence,property,age,other_installment_plans,housing,number_credits,job,people_liable,telephone,foreign_worker,credit_risk
0,no checking account,18,all credits at this bank paid back duly,car (used),1049,unknown/no savings account,< 1 yr,< 20,female : non-single or male : single,none,>= 7 yrs,car or other,21,none,for free,1,skilled employee/official,0 to 2,no,no,good
1,no checking account,9,all credits at this bank paid back duly,others,2799,unknown/no savings account,1 <= ... < 4 yrs,25 <= ... < 35,male : married/widowed,none,1 <= ... < 4 yrs,unknown / no property,36,none,for free,2-3,skilled employee/official,3 or more,no,no,good
2,... < 0 DM,12,no credits taken/all credits paid back duly,retraining,841,... < 100 DM,4 <= ... < 7 yrs,25 <= ... < 35,female : non-single or male : single,none,>= 7 yrs,unknown / no property,23,none,for free,1,unskilled - resident,0 to 2,no,no,good
3,no checking account,12,all credits at this bank paid back duly,others,2122,unknown/no savings account,1 <= ... < 4 yrs,20 <= ... < 25,male : married/widowed,none,1 <= ... < 4 yrs,unknown / no property,39,none,for free,2-3,unskilled - resident,3 or more,no,yes,good
4,no checking account,12,all credits at this bank paid back duly,others,2171,unknown/no savings account,1 <= ... < 4 yrs,< 20,male : married/widowed,none,>= 7 yrs,car or other,38,bank,rent,2-3,unskilled - resident,0 to 2,no,yes,good
